# 00e. Filter Relevansi Artikel (untuk Model B_2 dan C)

Filter Tugas 1 hanya meminta satu dari ~45 kata kunci di mana pun dalam isi artikel, sehingga artikel gaya hidup,
keuangan pribadi, dan halaman afiliasi ikut lolos. Notebook ini menerapkan aturan yang lebih ketat dan **dibekukan**
(tidak disetel terhadap hasil model) pada `articles_clean.csv`. Hasilnya `articles_filtered.csv`, yang dipakai
Model B_2 dan C. `articles_clean.csv` tidak diubah, sehingga Model B_1 (hasil Tugas 2) tetap dapat direproduksi.

Aturan:
1. Buang URL non-berita: `/select/`, `/quotes/`, `/advertorial/`, `/guide/`, `/video/`, `/pro/`, serta halaman harga saham dan *CNBC Daily Open*.
2. Normalisasi judul: hapus akhiran ` - CNBC` (muncul sejak 2026).
3. Pada **judul + lead** (bukan isi penuh) artikel harus memuat minimal satu kata kunci *inti* (perang, sanksi, tarif, kurs, suku bunga, inflasi, dll.) atau minimal dua kata kunci *umum*.

In [ ]:
import re
from pathlib import Path

import pandas as pd

DATA = Path("../data") if Path("../data").exists() else Path("data")
art = pd.read_csv(DATA / "processed" / "articles_clean.csv")
print(f"Artikel bersih: {len(art)}")

In [ ]:
CORE = r"""wars?|military|troops?|missiles?|invasion|invade|ceasefire|cease-fire|nato|nuclear|sanctions?|embargo|tariffs?|trade (?:war|deal|talks?|tensions?)|export controls?|geopolit\w*|ukrain\w*|russia\w*|putin|kremlin|israel\w*|gaza|hamas|hezbollah|iran\w*|houthi\w*|red sea|taiwan|north korea|middle east|opec\w*|crude|brent|oil (?:prices?|shock|supply|market\w*|output|production|exports?)|dollar|greenback|currency|currencies|forex|yuan|yen|rupiah|treasur(?:y|ies) yields?|bond yields?|yields?|fed|federal reserve|powell|warsh|central banks?|rate (?:cuts?|hikes?|decisions?)|interest rates?|inflation|cpi|recession|gdp|imf|world bank|brics|safe[- ]haven|supply chains?|debt ceiling|jobs report|payrolls|bank of (?:japan|england|canada)|ecb|pboc|indonesia\w*|coup|regime|airstrikes?|election|elections|afghanistan|syria|venezuela|china|chinese|beijing"""
WEAK = r"""strikes?|attacks?|conflict|economy|economic|economists?|gold|shipping|default|shutdown|stimulus|unemployment|trump|biden|white house|congress|g7|g20|summit|euro|energy|oil|gas|prices?|markets?|stocks?|trade|bank|debt|tax(?:es)?|president|government|minister|leaders?|security|defense|treasury|europe\w*|japan\w*|india\w*|saudi|uk|britain|u\.k\.|germany|french|france"""
RX_CORE = re.compile(r"\b(?:%s)\b" % CORE, re.I)
RX_WEAK = re.compile(r"\b(?:%s)\b" % WEAK, re.I)
URL_BURUK = r"cnbc\.com/(?:select|quotes|advertorial|guide|video|pro)/"
JUDUL_BURUK = r"^Check out .* stock price|^CNBC Daily Open"


def n_unik(rx, teks):
    return len({m.lower() for m in rx.findall(teks)})


art["clean_title"] = art["clean_title"].fillna("").str.replace(r"\s*-\s*CNBC$", "", regex=True)
kepala = art["clean_title"] + " " + art["lead_paragraph"].fillna("")
buruk = art["url"].str.contains(URL_BURUK, regex=True) | art["clean_title"].str.contains(JUDUL_BURUK, regex=True)
n_core = kepala.apply(lambda s: n_unik(RX_CORE, s))
n_weak = kepala.apply(lambda s: n_unik(RX_WEAK, s))
art["relevan"] = ((n_core >= 1) | (n_weak >= 2)) & ~buruk

print(f"Dibuang karena URL/judul non-berita : {int(buruk.sum())}")
print(f"Dibuang karena kata kunci            : {int((~buruk & ~art['relevan']).sum())}")
print(f"Dipertahankan                        : {int(art['relevan'].sum())} ({art['relevan'].mean():.1%})")
(art.assign(tahun=art["date"].str[:4]).groupby("tahun")["relevan"].agg(["size", "mean"]).round(3))

In [ ]:
print("DIBUANG:"); print(art.loc[~art.relevan, "clean_title"].sample(15, random_state=0).to_string())
print(); print("DIPERTAHANKAN:"); print(art.loc[art.relevan, "clean_title"].sample(15, random_state=0).to_string())

In [ ]:
out = art[art.relevan].drop(columns="relevan")
out.to_csv(DATA / "processed" / "articles_filtered.csv", index=False)
print(f"Tersimpan: {len(out)} artikel -> data/processed/articles_filtered.csv")